# 02b — Protocol v2: small validation diagnostic

**Purpose:** check answer stopping, extraction, and raw output quality before another benchmark or training run. The first baseline exposed repetition and format-scoring failures; its completed results remain protocol v1.

This notebook evaluates **10 fixed validation questions × 2 prompt modes = 20 predictions**. It has **no full-benchmark step**. The runner cannot evaluate more than 20 validation questions (40 predictions). Passing automated checks never launches a larger run.

**Kaggle:** enable Internet, select a GPU, and attach **Notebook 01's saved data output** using Add Input. You do not need to regenerate that bundle or attach the old baseline output. Run All, save the completed version with outputs, and inspect the raw-answer report.

The base model, model revision, prompts, three-shot demonstrations, greedy decoding, and 512-token safety cap are unchanged. The intervention is measurement: answer-boundary stopping and explicitly documented numeric normalization. A higher score here is **not evidence that the unchanged model learned maths**.

## 1. Load the shared code from GitHub

The notebook is a launcher; shared logic and the protocol live in GitHub. Each run records the exact commit. Normal code/config updates arrive by rerunning this cell.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPOSITORY = "https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git"
PROJECT_REF = os.environ.get("MATH_PROJECT_REF", "main")
if PROJECT_REF.startswith("-"):
    raise ValueError("Use a branch, tag, or full commit SHA as MATH_PROJECT_REF")
PROJECT = Path(tempfile.mkdtemp(prefix="math-project-"))
subprocess.run(["git", "init", "--quiet", str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "remote", "add", "origin", REPOSITORY], check=True)
subprocess.run(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", PROJECT_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "--quiet", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPOSITORY)
print("Code commit:", CODE_COMMIT)
print("Project checkout:", PROJECT)

## 2. Verify the original data bundle and inspect the diagnostic plan

This uses the saved validation split and training demonstrations. Neither the test datasets nor a new random split are used for generation. The first ten validation IDs come from the original deterministic split order; they are not selected by correctness.

We retain the same model and prompts so the effects of stopping/extraction can be examined without also changing the prompt template or switching models.

In [ ]:
import hashlib
import shutil

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings before continuing.")
_ = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=True)
INPUT_ROOT = Path(os.environ.get("MATH_BUNDLE_PATH", "/kaggle/input"))
BUNDLE = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_bundle.py"), str(INPUT_ROOT)], text=True).strip())
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/verify_bundle.py"), str(BUNDLE)], check=True)
PROTOCOL_PATH = PROJECT / "configs/protocol_v2_diagnostic.json"
PROTOCOL = json.loads(PROTOCOL_PATH.read_text())
SOURCE_MANIFEST = json.loads((BUNDLE / "manifest.json").read_text())
BUNDLE_HASH = hashlib.sha256((BUNDLE / "manifest.json").read_bytes()).hexdigest()
PROTOCOL_HASH = hashlib.sha256(PROTOCOL_PATH.read_bytes()).hexdigest()
OUTPUT = Path("/kaggle/working") / f"protocol-v2-{BUNDLE_HASH[:12]}-{PROTOCOL_HASH[:8]}-{CODE_COMMIT[:8]}"
print("Verified bundle:", BUNDLE)
print("Pinned model:", SOURCE_MANIFEST["model"])
print("Protocol:", PROTOCOL["protocol_id"])
print("Scope:", PROTOCOL["scope"])
print("Scored predictions:", PROTOCOL["validation_questions"] * len(PROTOCOL["modes"]))
print("Maximum new tokens:", PROTOCOL["generation"]["max_new_tokens"])
print("Output:", OUTPUT)
print("There is no full-benchmark or training step in this notebook.")

## 3. Prepare the known inference environment

This uses the previously verified isolated PyTorch/CUDA setup. It does not replace Kaggle kernel packages. The first T4 is sufficient. Installing the environment and downloading weights are separate from evaluating the 20 scored prompts.

The original nonfatal `sitecustomize`/`wrapt` startup warning may still appear on some Kaggle images. The shared model loader may also print the cosmetic `torch_dtype` deprecation warning; neither changes the FP16 runtime or the diagnostic protocol.

In [ ]:
ENVIRONMENT = Path(tempfile.mkdtemp(prefix="math-v2-env-"))
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/setup_environment.py"),
                    "--environment", str(ENVIRONMENT), "--profile", "baseline"], check=True)
PYTHON = ENVIRONMENT / "bin" / "python"
probe = """
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable: check Kaggle GPU settings.')
x = torch.ones((32, 32), device='cuda', dtype=torch.float16)
assert (x @ x).sum().item() == 32768
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))
print('CUDA execution passed')
"""
_ = subprocess.run([str(PYTHON), "-I", "-c", probe], check=True)

## 4. Check the stopping and scoring contracts before model inference

These CPU tests use tiny synthetic fixtures, not SmolLM2 benchmark scores. They check that:

- The number `550` cannot be cut off at its initial `5`.
- A finished row stops independently while other batch rows continue.
- Existing demonstration answers in the input cannot trigger a stop.
- Later invented questions cannot supply the target answer.
- `$9` normalizes to 9 and remains incorrect when the reference is 39.
- A numeric prefix at the token cap is flagged incomplete instead of silently credited.

In [ ]:
_ = subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", "test_protocol_v2.py", "-b", "-v"],
                    cwd=PROJECT, check=True)

## 5. Run the 20-prediction validation diagnostic

Generation stops independently for each row when:

1. Its first `Final answer:` line receives a newline; malformed values are retained and reported as unparseable.
2. A new `Question:`/`### Question:` header appears, including inline headers. Its answer is excluded.
3. The pinned model EOS is generated.
4. The unchanged 512-token safety cap is reached.

EOS or a next-question boundary can finalize an answer without a newline. A bare numeric prefix at the token cap is marked incomplete because more digits may have been truncated.

Normalization accepts documented currency prefixes, limited wrappers, trailing periods, and listed unit words. It does not convert units, search for the reference answer, or select an arbitrary last number. Every raw completion is retained through the token where stopping was detected; a token can contain a few characters beyond the boundary.

In [ ]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/run_protocol_v2.py"),
                    "--bundle", str(BUNDLE), "--protocol", str(PROTOCOL_PATH),
                    "--output", str(OUTPUT)], check=True, cwd=PROJECT)
SUMMARY = json.loads((OUTPUT / "summary.json").read_text())
if not SUMMARY["execution_complete"]:
    raise RuntimeError("Diagnostic is incomplete; rerun this cell with the same settings to resume.")
print("Completed validation predictions:", SUMMARY["predictions"])
print("Raw-output review required:", SUMMARY["review_required"])
print("Full benchmark authorized:", SUMMARY["full_benchmark_authorized"])

## 6. Review the answers and save the diagnostic

The report separates **strict numeric format**, **normalized parseability**, and **numeric correctness across all questions**. Accuracy among parseable answers is also saved, but it is selection-biased and cannot stand in for overall maths ability.

Expand every validation result to check the question, reference, extracted value, target text, raw text, and stopping reason. A correct final answer does not establish that every intermediate calculation is correct. Correcting extraction can change the score without changing the model.

Even if there are no quality alarms, this is a development diagnostic—not final test accuracy or permission to launch a full benchmark. Save this notebook with outputs and share its summary/report and raw predictions for review before the next experiment.

In [ ]:
from IPython.display import HTML, display
RESULT = json.loads((OUTPUT / "result_manifest.json").read_text())
for name, expected in RESULT["files_sha256"].items():
    if hashlib.sha256((OUTPUT / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError(f"Diagnostic result checksum mismatch: {name}")
display(HTML((OUTPUT / "report.html").read_text()))
print("Quality alarms:", SUMMARY["quality_alarms"] or "None automatically flagged; raw-answer review is still required.")
ARCHIVE = Path(shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT))
print("Diagnostic ZIP:", ARCHIVE)
print("Save Version with outputs. This notebook ends here: no full benchmark and no training.")